# Fine-tune Qwen3-VL-2B with LoRA on CORD (approach C)

Runs top to bottom on Kaggle's free **GPU T4 x2** (Internet on) or a single Colab T4. It:

1. clones the public repo and downloads CORD-v2 (train/validation/test) and the SROIE test split,
2. trains a LoRA adapter on the 800 CORD training receipts (`notebooks/train_lora.py`, one
   process per GPU), checkpointing as it goes,
3. evaluates the fine-tuned model with the same code as the laptop runs (CORD validation + test,
   SROIE, the four corruptions), plus T4 latency for the zero-shot model and Donut, as one
   queue per GPU,
4. zips predictions, training log and adapter into `results.zip` in the working directory.

**Bringing results back:** download `results.zip`, unzip it in the repo root (it contains
`results/...` and `adapters/qwen3vl_lora/`), then run `make report`.

**Resuming:** checkpoints go to `ckpt/` every 25 optimizer steps and every eval appends one line per
document, so re-running the notebook in the same session continues where it stopped. To continue
in a new Kaggle session, add the previous version's output as an input and set `RESUME_DIR` to its
`ckpt` folder.

In [ ]:
import os, sys, json, time, shutil, subprocess, threading, zipfile
from pathlib import Path

DRY_RUN = os.environ.get("DRY_RUN") == "1"  # tiny random model, 2 steps: checks the code path on a laptop
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working" if ON_KAGGLE else "/content" if Path("/content").exists() else os.environ.get("WORK", "."))
REPO_URL = "https://github.com/egkiastas/doc-extraction-vlm"
REPO = WORK / "doc-extraction-vlm"
CKPT, ADAPTER = WORK / "ckpt", WORK / "adapter"
RESUME_DIR = ""          # e.g. "/kaggle/input/<previous-output>/ckpt"

EPOCHS, LR, BATCH, LORA_R = 2, 2e-4, 8, 16   # BATCH is the effective batch over all GPUs
RUN_SROIE = True
RUN_ROBUSTNESS = True
RUN_ZERO_SHOT_T4 = True  # zero-shot on the T4 too: GPU latency for approach B

# Estimates for Kaggle's 2 x T4, not measurements: training assumes ~3 s per receipt per GPU at
# 1.3 MP (fp16 autocast, gradient checkpointing) split over two DDP processes; evals assume ~25
# generated tokens/s and run as two queues, one per GPU. On one GPU, roughly double both.
STAGES = [
    ("setup + data download (~2.3 GB)", 6),
    ("LoRA fine-tuning, 800 receipts x 2 epochs, 2 GPUs", 45),
    ("evals in two GPU queues (fine-tuned: CORD val+test, SROIE,", 50),
    ("  4 corruptions x 40; zero-shot CORD test; Donut 20)", 0),
]
print(f"{'stage':60s} est. minutes")
for name, minutes in STAGES:
    print(f"{name:60s} {minutes or '':>5}")
print(f"{'total':60s} {sum(m for _, m in STAGES):>5d}   (~{sum(m for _, m in STAGES) / 60:.1f} h)")

In [ ]:
def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not DRY_RUN:
    if not REPO.exists():
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
    # Kaggle/Colab already ship torch, pydantic, numpy, scipy, pillow, pyarrow, matplotlib.
    sh(f"{sys.executable} -m pip install -q transformers==5.24.0 peft==0.22.0 accelerate==1.16.0 zss==1.2.0 "
       "pypdfium2 pytesseract 'huggingface-hub>=1.0,<2'")
    sh(f"{sys.executable} -m pip install -q --no-deps -e {REPO}")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
N_GPU = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", N_GPU,
      torch.cuda.get_device_name(0) if N_GPU else "")

In [ ]:
t0 = time.time()
if not DRY_RUN:
    sh(f"{sys.executable} -m docextract.data --train")
print(f"data ready in {time.time() - t0:.0f}s")

In [ ]:
# Training runs as a separate script, one process per GPU (DDP). Don't train inside the notebook
# kernel: with two GPUs visible, Trainer falls back to DataParallel, which breaks Qwen3-VL.
if RESUME_DIR and not CKPT.exists():
    shutil.copytree(RESUME_DIR, CKPT)
if (ADAPTER / "adapter_config.json").exists():
    print("adapter already trained, skipping")
else:
    t0 = time.time()
    procs = max(1, N_GPU) if not DRY_RUN else int(os.environ.get("DRY_RUN_PROCS", "1"))
    sh(f"{sys.executable} -m torch.distributed.run --nnodes 1 --master_addr 127.0.0.1 "
       f"--master_port 29511 --nproc_per_node {procs} "
       f"notebooks/train_lora.py --work {WORK} --epochs {EPOCHS} --lr {LR} --batch {BATCH} "
       f"--lora-r {LORA_R} --log-dir {REPO / 'results' / 'finetune'}" + (" --dry-run" if DRY_RUN else ""))
    print(f"training stage took {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Same evaluation code as the laptop runs; each command appends to results/predictions/...
# and skips documents already done, so a re-run resumes. Commands are spread over one queue per
# GPU (largest first, to the least-loaded queue). Both queues share the 4 vCPUs, so the T4
# latencies are measured with the other GPU busy; CPU contention can only make them slower.
os.environ["VLM_ADAPTER"] = str(ADAPTER)
if DRY_RUN:
    os.environ["VLM_MODEL"] = "trl-internal-testing/tiny-Qwen3VLForConditionalGeneration"
    os.environ["VLM_MAX_NEW_TOKENS"] = "16"
limit = " --limit 2" if DRY_RUN else ""
jobs = [(25, "--approach qwen3vl_lora --split cord-test"),   # (estimated minutes, args)
        (25, "--approach qwen3vl_lora --split cord-validation")]
if RUN_SROIE:
    jobs.append((12, "--approach qwen3vl_lora --split sroie-test"))
if RUN_ROBUSTNESS:
    jobs += [(5, f"--approach qwen3vl_lora --split cord-test --corruption {c}"
                 + ("" if DRY_RUN else " --limit 40")) for c in ["rotate", "blur", "lowres", "jpeg"]]
if N_GPU:  # T4 latency for approaches B and Donut; files end in -cuda
    if RUN_ZERO_SHOT_T4:
        jobs.append((30, "--approach qwen3vl --split cord-test --device cuda"))
    jobs.append((2, "--approach donut --split cord-test --device cuda --limit 20"))

queues = [[] for _ in range(max(1, N_GPU))]
for minutes, args in sorted(jobs, reverse=True):
    min(queues, key=lambda q: sum(m for m, _ in q)).append((minutes, args))

def run_queue(gpu, queue, failed):
    env = dict(os.environ, **({"CUDA_VISIBLE_DEVICES": str(gpu)} if N_GPU else {}))
    for _, args in queue:
        cmd = f"{sys.executable} -m evaluation.run_eval {args}{limit}"
        print(f"[gpu{gpu}] $ {cmd}", flush=True)
        p = subprocess.Popen(cmd, shell=True, env=env, text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            if line.strip() and not any(w in line for w in ("arning", "warn(", "Loading weights")):
                print(f"[gpu{gpu}] {line}", end="", flush=True)
        if p.wait():
            failed.append(args)

t0, failed = time.time(), []
threads = [threading.Thread(target=run_queue, args=(g, q, failed)) for g, q in enumerate(queues)]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"evals took {(time.time() - t0) / 60:.1f} min")
assert not failed, f"failed (re-run this cell to resume): {failed}"

In [ ]:
# Copy the adapter next to the results and zip both.
dest = REPO / "adapters" / "qwen3vl_lora"
if ADAPTER.exists():
    shutil.copytree(ADAPTER, dest, dirs_exist_ok=True)
out = WORK / "results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["results/predictions", "results/finetune", "adapters"]:
        for f in sorted((REPO / base).rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(REPO))
print(out, f"{out.stat().st_size / 1e6:.1f} MB")